In [10]:
%pip install torch transformers pandas scikit-learn

  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl (20.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


필수 라이브러리 로드 및 환경 세팅

In [11]:
# 필요한 라이브러리 불러오기
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

# GPU(CUDA) 또는 CPU 설정 (애플 실리콘 Mac의 경우 'mps', 윈도우/리눅스는 'cuda')
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🚀 현재 사용 중인 디바이스: {device}")

🚀 현재 사용 중인 디바이스: mps


실제 데이터(ratings_train.txt) 불러오기

In [12]:
# 1. 팀원 1이 전처리한 파일 불러오기
df = pd.read_csv('preprocessed_train_data.csv')

# 빠른 테스트를 위해 5,000개만 샘플링 (전체 학습 시 이 줄 주석 처리)
df_sample = df.sample(n=5000, random_state=42).reset_index(drop=True)

# 2. 학습용(Train) 80%, 검증용(Val) 20%로 데이터 쪼개기
train_df, val_df = train_test_split(df_sample, test_size=0.2, random_state=42)

print(f"📊 학습 데이터 {len(train_df)}개, 모의고사(검증) 데이터 {len(val_df)}개 준비 완료!")

📊 학습 데이터 4000개, 모의고사(검증) 데이터 1000개 준비 완료!


토크나이저 및 PyTorch 데이터셋 세팅. Tokenizer(사람의 말을 AI가 이해할 수 있는 숫자 암호로 번역해 주는 사전)만들고 모델에 넣을 준비

In [13]:
# 3. 모델 이름: 한국어 구어체에 강력한 최신 KcBERT 사용
model_name = 'beomi/kcbert-base' 
tokenizer = BertTokenizer.from_pretrained(model_name)

# 4. PyTorch Dataset 클래스 정의
class NsmcDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data['clean_text'].iloc[index]) # 팀원 1의 정제 열 사용
        label = self.data['label'].iloc[index]

        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 5. Train / Val DataLoader 각각 생성
train_loader = DataLoader(NsmcDataset(train_df, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(NsmcDataset(val_df, tokenizer), batch_size=16, shuffle=False)

print("✅ 토크나이징 및 Train/Val DataLoader 세팅 완료!")

✅ 토크나이징 및 Train/Val DataLoader 세팅 완료!


KcBERT 모델 로드 및 학습(Train) 진행
뼈대만 있는 KoBERT 모델을 가져와서, 우리가 만든 데이터를 총 3번(epochs=3) 반복해서 읽히면서 "이 단어 조합은 악플(0)이야, 이건 칭찬(1)이야" 하고 뇌를 훈련(Fine-tuning)시키기

In [ ]:
# 6. 분류용 모델 로드
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3 

print("🔥 딥러닝 모델 학습 및 검증 시작!\n")

# 7. 학습 및 검증 루프
for epoch in range(epochs):
    # --- [A] 학습 (Train) 단계 ---
    model.train()
    train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # --- [B] 검증 (Evaluation) 단계 ---
    model.eval()
    val_loss, correct, total = 0, 0, 0
    
    with torch.no_grad(): # 검증 시에는 역전파 금지
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            
            # 정확도 계산
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"🎯 Epoch [{epoch+1}/{epochs}] 완료!")
    print(f"   - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"   - 🏆 Val Accuracy: {val_accuracy * 100:.2f}%\n")

print("🎉 3단계: 검증 루프가 포함된 최종 학습 파이프라인이 완벽하게 끝났습니다!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8321.53it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

🔥 딥러닝 모델 학습 및 검증 시작!

🎯 Epoch [1/3] 완료!
   - Train Loss: 0.4703 | Val Loss: 0.3739
   - 🏆 Val Accuracy: 83.60%

🎯 Epoch [2/3] 완료!
   - Train Loss: 0.2571 | Val Loss: 0.3691
   - 🏆 Val Accuracy: 84.10%

